# Lecture 2 - Python, pandas, and an Excel workbook

**Question:** How do we turn a formatted report into a table we can trust?

Lec 1 used prepared data. Today we build it. We retain the **passenger-car section** from every worksheet, not trucks or other vehicle categories.

**Sequence:** Python → selection → one sheet → reusable function → workbook → validation.

Use a Python environment with pandas and openpyxl. Installation is a before-class task. Run cells in order; restart and run all to reproduce.

In [1]:
# Google Colab only: download this lecture's data and helper files, then work inside that folder.
# Running locally? Start Jupyter in the lecture folder and this cell does nothing.
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('requirements.txt'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/DrFarrukh/teaching.git /content/teaching
    !git -C /content/teaching sparse-checkout set courses/data-analysis-and-visualization/lectures/lecture-02
    os.chdir('/content/teaching/courses/data-analysis-and-visualization/lectures/lecture-02')
    !pip -q install -r requirements.txt
    print('Working folder:', os.getcwd())

## 1. Python essentials
An assignment gives a value a name. Strings are text; integers are whole numbers. A list stores an ordered collection. `=` assigns; `==` compares. Predict the last expression before running.


In [2]:
model = "Honda Civic"
sales = [418, 630, 305]
total = sum(sales)
print(model, total)
total / len(sales)


Honda Civic 1353


451.0

List positions start at zero. A slice includes the start and excludes the stop. A function call uses parentheses; indexing uses square brackets. What are `sales[0]` and `sales[0:2]`?


In [3]:
print(sales[0])
print(sales[0:2])
print(total > 1000)
print(type(model), type(total))


418
[418, 630]
True
<class 'str'> <class 'int'>


### Load a library; create a small table
A dictionary pairs column names with values. pandas calls a table a **DataFrame**. Keep this tiny example visible while learning selection.


In [4]:
import pandas as pd

example = pd.DataFrame({
    "model": ["A", "B", "C"],
    "sales": [100, 0, 150],
    "production": [110, 20, 140]
})
example


,model,sales,production
0,A,100,110
1,B,0,20
2,C,150,140


## 2. Select columns and rows
One column gives a Series; a list of column names gives a DataFrame. Predict their shapes.


In [5]:
example["sales"]


0    100
1      0
2    150
Name: sales, dtype: int64

In [6]:
example[["model", "sales"]]


,model,sales
0,A,100
1,B,0
2,C,150


### Positional selection: `iloc`
For a table with $n$ rows, positions are $0,1,\ldots,n-1$.
The slice $[a:b]$ selects positions $a\leq i<b$. In bounds, it contains $b-a$ positions.
`iloc[rows, columns]` uses positions on both axes.


In [7]:
example.iloc[0:2, 0:2]


,model,sales
0,A,100
1,B,0


### Conditional selection: `loc`
A condition produces one Boolean value per row. `loc` uses that mask to retain matching rows and can select named columns at the same time.


In [8]:
mask = example["sales"] > 0
print(mask)
example.loc[mask, ["model", "sales"]]


0     True
1    False
2     True
Name: sales, dtype: bool


,model,sales
0,A,100
2,C,150


Use `&` for elementwise AND and parentheses around each condition. `.loc` label slices include the ending label; `.iloc` position slices exclude the stop. After filtering, index labels need not equal row positions.


In [9]:
example.loc[(example["sales"] > 0) & (example["production"] > 120)]


,model,sales,production
2,C,150,140


### Open the source workbook
The workbook sits beside this notebook. `header=None` prevents pandas from treating the report title as column names. The openpyxl warning about a header/footer concerns print decoration, not data cells.


In [10]:
from pathlib import Path

file = Path("five-years-12.xlsx")
workbook = pd.ExcelFile(file)
print(workbook.sheet_names)

['2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']


In [11]:
sheet_name = "2007-08"
# sheet_name = workbook.sheet_names[-1]
raw = pd.read_excel(workbook, sheet_name=sheet_name, header=None)
print(raw.shape)
raw.iloc[:60]


(187, 16)


.../site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,NaN,NaN,NaN,2007-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007-08
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,PASSENGER CARS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,1300 and Above cc,NaN,Jul'07,Aug'07,Sep'07,Oct' 07,Nov'07,Dec'07,Jan'08,Feb'08,Mar'08,Apr'08,May'08,June'08,Cumulative
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,Honda Civic,Prod.,554,533,405,662,423,363,477,598,332,419,418,629,5813
9,NaN,NaN,Sale,418,630,305,581,348,315,521,579,449,527,417,672,5762


In [12]:
# print(raw[1].value_counts())
truck_rows = raw.index[raw[1] == "TRUCKS"]
truck_rows.to_list()

[55]

In [13]:
column = raw[1]

column = column.astype("string")
column = column.str.strip()
column = column.str.upper()

mask = column.eq("TRUCKS")

truck_rows = raw.index[mask]
truck_rows.to_list()

[55]

In [14]:
truck_rows = raw.index[raw[1].astype("string").str.strip().str.upper().eq("TRUCKS")]

print("Pandas row number:", truck_rows.tolist())

Pandas row number: [55]


In [15]:
import pandas as pd

demo = pd.DataFrame({
    "model_group": ["  Honda   Civic  ",
                    None,
                    "Suzuki   Alto",
                    None],
    "measure": [" Prod. ",
                " SALE ",
                "Prod.",
                " sale.  "]
})

demo

,model_group,measure
0,Honda Civic,Prod.
1,NaN,SALE
2,Suzuki Alto,Prod.
3,NaN,sale.


In [16]:
demo["model_group"] = demo["model_group"].ffill()
demo

,model_group,measure
0,Honda Civic,Prod.
1,Honda Civic,SALE
2,Suzuki Alto,Prod.
3,Suzuki Alto,sale.


In [17]:
demo["measure"].str.lower().map(repr)

0     ' prod. '
1      ' sale '
2       'prod.'
3    ' sale.  '
Name: measure, dtype: str

In [18]:
demo["measure"].str.strip().map(repr)

0    'Prod.'
1     'SALE'
2    'Prod.'
3    'sale.'
Name: measure, dtype: str

In [19]:
demo["measure"].str.rstrip(".").map(repr)

0     ' Prod. '
1      ' SALE '
2        'Prod'
3    ' sale.  '
Name: measure, dtype: str

In [20]:
demo["model_group"].str.split().map(repr)

0    ['Honda', 'Civic']
1    ['Honda', 'Civic']
2    ['Suzuki', 'Alto']
3    ['Suzuki', 'Alto']
Name: model_group, dtype: str

In [21]:
demo["model_group"].str.split().str.join(" ").map(repr)

0    'Honda Civic'
1    'Honda Civic'
2    'Suzuki Alto'
3    'Suzuki Alto'
Name: model_group, dtype: str

In [22]:
demo["model_group"] = (
    demo["model_group"]
    .ffill()
    .str.split()
    .str.join(" ")
)

demo["measure"] = (
    demo["measure"]
    .str.strip()
    .str.lower()
    .str.rstrip(".")
)

demo

,model_group,measure
0,Honda Civic,prod
1,Honda Civic,sale
2,Suzuki Alto,prod
3,Suzuki Alto,sale


### Pair exercise
Without a laptop, write code to select:
1. The first three rows and first two columns of `example`.
2. Model and production where sales equal zero.
3. The raw worksheet's Excel rows 9–13 and columns B–D.

Explain why `raw.iloc[8:13, 1:4]` has five rows and three columns.

**Namaz break.** After the break we transform the sheet.


## 3. One complete worksheet
### Select the passenger-car block
We inspected the source: passenger cars start at position 4; trucks start at 55. Month values occupy positions 3–14; cumulative total is at 15. These positions are valid for **2007–08 only**.

`copy()` makes a separate working table. Retain Excel row numbers before resetting any index.


In [23]:
# cars = raw.iloc[:55, 1:16]
# cars

In [24]:
cars = raw.iloc[5:55, 1:16].copy()
months = ["Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
          "Jan", "Feb", "Mar", "Apr", "May", "Jun"]
cars.columns = ["model_group", "measure"] + months + ["source_total"]
cars["source_row"] = cars.index + 1
cars


,model_group,measure,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr,May,Jun,source_total,source_row
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
6,1300 and Above cc,NaN,Jul'07,Aug'07,Sep'07,Oct' 07,Nov'07,Dec'07,Jan'08,Feb'08,Mar'08,Apr'08,May'08,June'08,Cumulative,7
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
8,Honda Civic,Prod.,554,533,405,662,423,363,477,598,332,419,418,629,5813,9
9,NaN,Sale,418,630,305,581,348,315,521,579,449,527,417,672,5762,10
10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
11,Honda City,Prod.,1046,960,1035,778,657,297,423,572,438,361,782,871,8220,12
12,NaN,Sale,611,1506,497,621,605,328,852,641,694,484,658,942,8439,13
13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14
14,Suzuki Liana,Prod.,353,199,514,439,306,10,30,175,280,145,96,58,2605,15


### Restore labels within this section
A merged model label appears once, above its sales row. Forward fill copies that label down. It is appropriate for **labels in this known block**, not for numeric sales. Keep category/header labels until measurement rows have been selected.


In [25]:
cars["model_group"] = cars["model_group"].ffill()
cars["model_group"] = cars["model_group"].str.split().str.join(" ")
cars["measure"] = cars["measure"].str.strip().str.lower().str.rstrip(".")
cars

,model_group,measure,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr,May,Jun,source_total,source_row
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
6,1300 and Above cc,NaN,Jul'07,Aug'07,Sep'07,Oct' 07,Nov'07,Dec'07,Jan'08,Feb'08,Mar'08,Apr'08,May'08,June'08,Cumulative,7
7,1300 and Above cc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
8,Honda Civic,prod,554,533,405,662,423,363,477,598,332,419,418,629,5813,9
9,Honda Civic,sale,418,630,305,581,348,315,521,579,449,527,417,672,5762,10
10,Honda Civic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
11,Honda City,prod,1046,960,1035,778,657,297,423,572,438,361,782,871,8220,12
12,Honda City,sale,611,1506,497,621,605,328,852,641,694,484,658,942,8439,13
13,Honda City,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14
14,Suzuki Liana,prod,353,199,514,439,306,10,30,175,280,145,96,58,2605,15


### Select measurement rows; exclude totals
`isin()` checks membership. `~` negates a Boolean mask. Including subtotals alongside their components would double count units.


In [26]:
cars["measure"].value_counts()

measure
prod    14
sale    14
Name: count, dtype: int64

In [27]:
valid_measures = ["prod", "sale"]
cars = cars.loc[cars["measure"].isin(valid_measures)].copy()
is_total = cars["model_group"].str.contains("TOTAL", case=False, regex=False)
cars = cars.loc[~is_total].copy()
cars[["model_group", "measure", "Jul", "source_row"]].head()


,model_group,measure,Jul,source_row
8,Honda Civic,prod,554,9
9,Honda Civic,sale,418,10
11,Honda City,prod,1046,12
12,Honda City,sale,611,13
14,Suzuki Liana,prod,353,15


In [28]:
cars

,model_group,measure,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr,May,Jun,source_total,source_row
8,Honda Civic,prod,554,533,405,662,423,363,477,598,332,419,418,629,5813,9
9,Honda Civic,sale,418,630,305,581,348,315,521,579,449,527,417,672,5762,10
11,Honda City,prod,1046,960,1035,778,657,297,423,572,438,361,782,871,8220,12
12,Honda City,sale,611,1506,497,621,605,328,852,641,694,484,658,942,8439,13
14,Suzuki Liana,prod,353,199,514,439,306,10,30,175,280,145,96,58,2605,15
15,Suzuki Liana,sale,318,613,386,162,192,175,180,186,208,196,148,219,2983,16
17,Toyota Corolla,prod,3745,3404,2826,2543,2702,1690,1888,2654,2944,3091,3034,3151,33672,18
18,Toyota Corolla,sale,3414,3711,2546,2531,2637,1141,2485,2704,3054,3054,2923,3440,33640,19
25,Suzuki Cultus,prod,709,3166,3019,3131,2047,1739,1524,2040,2477,2980,2966,1864,27662,26
26,Suzuki Cultus,sale,389,3398,2340,2838,1712,1674,2080,2204,2707,2795,2221,3205,27563,27


In [29]:
cars["source_total"] = pd.to_numeric(cars["source_total"], errors="coerce")
measure_names = {"prod": "Production",
                 "sale": "Sales"}
cars["measure"] = cars["measure"].replace(measure_names)
print(cars.shape)
cars["measure"].value_counts()
cars


(20, 16)


,model_group,measure,Jul,Aug,Sep,Oct,Nov,Dec,Jan,Feb,Mar,Apr,May,Jun,source_total,source_row
8,Honda Civic,Production,554,533,405,662,423,363,477,598,332,419,418,629,5813,9
9,Honda Civic,Sales,418,630,305,581,348,315,521,579,449,527,417,672,5762,10
11,Honda City,Production,1046,960,1035,778,657,297,423,572,438,361,782,871,8220,12
12,Honda City,Sales,611,1506,497,621,605,328,852,641,694,484,658,942,8439,13
14,Suzuki Liana,Production,353,199,514,439,306,10,30,175,280,145,96,58,2605,15
15,Suzuki Liana,Sales,318,613,386,162,192,175,180,186,208,196,148,219,2983,16
17,Toyota Corolla,Production,3745,3404,2826,2543,2702,1690,1888,2654,2944,3091,3034,3151,33672,18
18,Toyota Corolla,Sales,3414,3711,2546,2531,2637,1141,2485,2704,3054,3054,2923,3440,33640,19
25,Suzuki Cultus,Production,709,3166,3019,3131,2047,1739,1524,2040,2477,2980,2966,1864,27662,26
26,Suzuki Cultus,Sales,389,3398,2340,2838,1712,1674,2080,2204,2707,2795,2221,3205,27563,27


### Wide → long: reorganize, do not aggregate
For $G$ model groups, $Q$ measures and $M$ months, a complete block has
$$R_{\text{wide}}=GQ,\qquad R_{\text{long}}=GQM.$$
More generally, `melt` produces $M R_{\text{wide}}$ rows even when some values are missing.

Each monthly cell $x_{i,m}$ becomes a record $(i,m,x_{i,m})$. Trace Honda Civic / Sales / July before and after. The cumulative total stays an identifier for later checking; it is **not a thirteenth month**.


In [30]:
print(months)
cars[months[0]]
# cars["Jul"]

['Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']


8      554
9      418
11    1046
12     611
14     353
15     318
17    3745
18    3414
25     709
26     389
28    1981
29    1320
31     302
32     302
39    1104
40    1054
42    2976
43    2690
45    1182
46    1139
Name: Jul, dtype: object

In [31]:
ids = ["model_group", "measure", "source_row", "source_total"]
long_data = cars.melt(id_vars=ids, value_vars=months,
                      var_name="month", value_name="units")
print(len(cars), len(long_data))
long_data.head(24)


20 240


,model_group,measure,source_row,source_total,month,units
0,Honda Civic,Production,9,5813,Jul,554
1,Honda Civic,Sales,10,5762,Jul,418
2,Honda City,Production,12,8220,Jul,1046
3,Honda City,Sales,13,8439,Jul,611
4,Suzuki Liana,Production,15,2605,Jul,353
5,Suzuki Liana,Sales,16,2983,Jul,318
6,Toyota Corolla,Production,18,33672,Jul,3745
7,Toyota Corolla,Sales,19,33640,Jul,3414
8,Suzuki Cultus,Production,26,27662,Jul,709
9,Suzuki Cultus,Sales,27,27563,Jul,389


### Numeric conversion must be auditable
Save the original value. `errors="coerce"` turns unparseable text into missing data; it does not justify silently discarding it. Inspect conversion failures. A genuine zero stays zero. Entirely blank measurement rows are retained for review.


In [32]:
long_data["raw_units"] = long_data["units"]
long_data["units"] = pd.to_numeric(long_data["raw_units"], errors="coerce")
bad = long_data["raw_units"].notna() & long_data["units"].isna()
print(bad.sum())
long_data.loc[bad, ["model_group", "month", "raw_units"]]

0


,model_group,month,raw_units


### Fiscal years cross calendar years
Let $s$ be the fiscal start year and $m$ the calendar month number:
$$y=s+\mathbf{1}(m\leq6).$$
Thus January in FY 2007–08 belongs to **2008**, not 2007. Dates use day 1 as a monthly label; this is not a daily transaction date.


In [33]:
month_numbers = {"Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11,
                 "Dec": 12, "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
                 "May": 5, "Jun": 6}
month_number = long_data["month"].map(month_numbers)
start_year = int(sheet_name[:4])
year = start_year + (month_number <= 6).astype(int)
year

0      2007
1      2007
2      2007
3      2007
4      2007
       ... 
235    2008
236    2008
237    2008
238    2008
239    2008
Name: month, Length: 240, dtype: int64

In [34]:
long_data["date"] = pd.to_datetime({"year": year, "month": month_number, "day": 1})
long_data["fiscal_year"] = sheet_name
long_data["source_sheet"] = sheet_name
long_data[["fiscal_year", "date", "model_group", "measure", "units"]].head()

,fiscal_year,date,model_group,measure,units
0,2007-08,2007-07-01,Honda Civic,Production,554
1,2007-08,2007-07-01,Honda Civic,Sales,418
2,2007-08,2007-07-01,Honda City,Production,1046
3,2007-08,2007-07-01,Honda City,Sales,611
4,2007-08,2007-07-01,Suzuki Liana,Production,353


### Check one sheet before scaling
`assert` stops execution if a required condition is false. The physical source key is sheet + source row + date. The analytical key is date + model group + measure.


In [35]:
assert len(long_data) == len(cars) * 12
assert not long_data.duplicated(["source_sheet", "source_row", "date"]).any()
print(long_data["date"].min(), long_data["date"].max())
print("Missing units:", long_data["units"].isna().sum())


2007-07-01 00:00:00 2008-06-01 00:00:00
Missing units: 0


## 4. Why fixed positions do not scale
Read another year. Compare the passenger-car heading, monthly headings, and the next vehicle section. Print a small slice, not the whole workbook.

We keep columns B and C as the label/measure columns because this workbook uses them consistently. We will detect **section boundaries and month positions**. This is a workbook-specific parser, not a universal Excel cleaner.


In [36]:
second = pd.read_excel(workbook, sheet_name="2025-26", header=None)
# second
second.iloc[4:16, 1:9]

,1,2,3,4,5,6,7,8
4,1300cc and Above,NaN,NaN,NaN,July'25,Aug'25,Sept'25,Oct'25
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Honda Cars (Civic & City),Prod,NaN,NaN,1169,968,1939,2351
7,NaN,Sale,NaN,NaN,1143,699,1977,2247
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Suzuki Swift,Prod,NaN,NaN,648,1591,1394,1078
10,NaN,Sale,NaN,NaN,522,1473,1403,1170
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,"Toyota (Corolla, Yaris...",Prod,NaN,NaN,2484,2614,2708,3809
13,NaN,Sale,NaN,NaN,2418,2553,2655,3742


### Locate the section by labels
Find the first boundary after PASSENGER CARS. `index[0]` selects the first match. Verify these assumptions rather than assuming every workbook follows them.


In [37]:
labels = second[1].astype("string").str.strip().str.upper()
start = labels.loc[labels == "PASSENGER CARS"].index[0]
boundary = labels.str.contains("LCVS", regex=False, na=False) | labels.eq("TRUCKS").fillna(False)
end = labels.loc[boundary & (labels.index > start)].index[0]
print(start, end)


2 60


### A small loop discovers the twelve month columns
`for` repeats an indented block. `if` chooses whether to run a block. `break` leaves the loop. Convert each candidate header to its first three lowercase letters. A cumulative label may also start with Jul: take the first twelve month matches and verify July–June order.


In [38]:
month_prefixes = ["jul", "aug", "sep", "oct", "nov", "dec",
                  "jan", "feb", "mar", "apr", "may", "jun"]
for row in range(start + 1, start + 12):
    header = second.iloc[row].astype("string").str.strip().str.lower().str[:3]
    month_cols = header.loc[header.isin(month_prefixes)].index.tolist()
    if len(month_cols) >= 12:
        month_cols = month_cols[:12]
        break
print(month_cols)
assert header.loc[month_cols].tolist() == month_prefixes


[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


## 5. Make the working recipe reusable
Let us introduce `def`: input is a worksheet name; output is its clean table. Local variables belong to this function call. `return` gives the result back.

The next cell is deliberately longer: it **collects the steps already taught**. Reveal it in blocks, rather than typing the whole function live. The added source-total rule takes the first numeric cell after the twelve months, matching this workbook's layout. Missing totals remain missing. No regular expressions, nested helper functions, or comprehensions are needed.


In [39]:
def clean_sheet(sheet_name):
    raw = pd.read_excel(workbook, sheet_name=sheet_name, header=None)
    labels = raw[1].astype("string").str.strip().str.upper()
    starts = labels.loc[labels == "PASSENGER CARS"].index
    assert len(starts) == 1, sheet_name
    start = starts[0]
    boundary = labels.str.contains("LCVS", regex=False, na=False) | labels.eq("TRUCKS").fillna(False)
    ends = labels.loc[boundary & (labels.index > start)].index
    assert len(ends) > 0, sheet_name
    end = ends[0]

    month_cols = []
    for row in range(start + 1, start + 12):
        header = raw.iloc[row].astype("string").str.strip().str.lower().str[:3]
        month_cols = header.loc[header.isin(month_prefixes)].index.tolist()
        if len(month_cols) >= 12:
            month_cols = month_cols[:12]
            break
    assert len(month_cols) == 12, sheet_name
    assert header.loc[month_cols].tolist() == month_prefixes, sheet_name

    cars = raw.iloc[start + 1:end, [1, 2] + month_cols].copy()
    cars.columns = ["model_group", "measure"] + months
    cars["source_row"] = cars.index + 1
    # The first numeric field after the month block is the cached annual total.
    totals = raw.iloc[start + 1:end, max(month_cols) + 1:].copy()
    for column in totals.columns:
        totals[column] = pd.to_numeric(totals[column], errors="coerce")
    cars["source_total"] = totals.bfill(axis=1).iloc[:, 0]

    cars["model_group"] = cars["model_group"].ffill()
    cars["model_group"] = cars["model_group"].str.split().str.join(" ")
    cars["measure"] = cars["measure"].str.strip().str.lower().str.rstrip(".")
    cars = cars.loc[cars["measure"].isin(valid_measures)].copy()
    is_total = cars["model_group"].str.contains("TOTAL", case=False, regex=False)
    cars = cars.loc[~is_total].copy()
    cars["measure"] = cars["measure"].replace(measure_names)

    result = cars.melt(id_vars=ids, value_vars=months,
                       var_name="month", value_name="units")
    result["raw_units"] = result["units"]
    result["units"] = pd.to_numeric(result["raw_units"], errors="coerce")
    month_number = result["month"].map(month_numbers)
    start_year = int(sheet_name[:4])
    year = start_year + (month_number <= 6).astype(int)
    result["date"] = pd.to_datetime({"year": year, "month": month_number, "day": 1})
    result["fiscal_year"] = sheet_name
    result["source_sheet"] = sheet_name
    assert len(result) == len(cars) * 12, sheet_name
    return result


### Check that packaging did not change the answer
Compare the function output with the first sheet we built step by step. This verifies the refactoring, not the truth of the source.


In [40]:
first_again = clean_sheet("2007-08")
pd.testing.assert_frame_equal(first_again, long_data, check_dtype=False)
print("Function matches the step-by-step result.")


Function matches the step-by-step result.


.../site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


### Convert the complete workbook
`tables` is a list of DataFrames. `append()` adds one table; `concat()` stacks them. `ignore_index=True` creates a fresh row index. Source row numbers are preserved separately. The function uses the shared mappings defined above; run the notebook in order.


In [41]:
tables = []
for sheet_name in workbook.sheet_names:
    table = clean_sheet(sheet_name)
    tables.append(table)
    print(sheet_name, len(table))

all_data = pd.concat(tables, ignore_index=True)
print("Combined records:", len(all_data))


2007-08 240


2008-09 240


.../site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
.../site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


2009-10 264


2010-11 264
2011-12 264


...

2018-19 192


2019-20 216
2020-21 240
2021-22 240


2022-23 240


2023-24 240


2024-25 288
2025-26 288
Combined records: 4560


## 6. Validate before saving
### Coverage, uniqueness and domain
Within each source measurement row, expect twelve monthly records. A valid analytical key is
$$K=(\text{date},\text{model group},\text{measure}).$$
Uniqueness means $\operatorname{count}(K)=1$. Production and sales must not be added together as one quantity.


In [42]:
assert set(all_data["source_sheet"]) == set(workbook.sheet_names)
assert set(all_data["measure"]) == {"Production", "Sales"}
assert not all_data.duplicated(["date", "model_group", "measure"]).any()
assert all_data["model_group"].notna().all()
assert all_data["units"].dropna().ge(0).all()
assert all_data["units"].dropna().mod(1).eq(0).all()
all_data.groupby("source_sheet").size()

source_sheet
2007-08    240
2008-09    240
2009-10    264
2010-11    264
2011-12    264
2012-13    264
2013-14    240
2014-15    240
2015-16    216
2016-17    192
2017-18    192
2018-19    192
2019-20    216
2020-21    240
2021-22    240
2022-23    240
2023-24    240
2024-25    288
2025-26    288
dtype: int64

### Missingness and conversion failures
$$r_{\text{missing}}=\frac{\sum_{i=1}^{N}\mathbf{1}(x_i\text{ is missing})}{N}.$$
Report the denominator. Missingness is not a zero; zero is a recorded value. A source zero may still require interpretation.


In [43]:
conversion_failed = all_data["raw_units"].notna() & all_data["units"].isna()
conversion_issues = all_data.loc[conversion_failed].copy()
print("Conversion failures:", len(conversion_issues))
print("Missing fraction:", all_data["units"].isna().mean())
print("Recorded zeros:", all_data["units"].eq(0).sum())
conversion_issues.head()

Conversion failures: 0
Missing fraction: 0.0
Recorded zeros: 645


,model_group,measure,source_row,source_total,month,units,raw_units,date,fiscal_year,source_sheet


### Reconcile each measurement row against the source
For complete monthly observations,
$$\widehat T_{g,q,y}=\sum_{m=1}^{12}x_{g,q,y,m},\qquad
\Delta_{g,q,y}=\widehat T_{g,q,y}-T^{\mathrm{source}}_{g,q,y}.$$
We require $\Delta=0$ for integer unit counts. If a month or source total is missing, mark the row **not checkable**. pandas normally skips missing values in sums; `min_count=12` prevents an incomplete year from looking complete.

Agreement checks arithmetic and extraction. It does not establish market coverage or whether values are final actual observations.


In [44]:
keys = ["source_sheet", "source_row", "model_group", "measure"]
grouped = all_data.groupby(keys)
checks = grouped["units"].sum(min_count=12).to_frame("monthly_sum")
checks["months_present"] = grouped["units"].count()
checks["source_total"] = grouped["source_total"].first()
checks["difference"] = checks["monthly_sum"] - checks["source_total"]
checks = checks.reset_index()
checks.head()


,source_sheet,source_row,model_group,measure,monthly_sum,months_present,source_total,difference
0,2007-08,9,Honda Civic,Production,5813,12,5813.0,0.0
1,2007-08,10,Honda Civic,Sales,5762,12,5762.0,0.0
2,2007-08,12,Honda City,Production,8220,12,8220.0,0.0
3,2007-08,13,Honda City,Sales,8439,12,8439.0,0.0
4,2007-08,15,Suzuki Liana,Production,2605,12,2605.0,0.0


In [45]:
checks["status"] = "not checkable"
checkable = checks["difference"].notna()
checks.loc[checkable & checks["difference"].eq(0), "status"] = "match"
checks.loc[checkable & checks["difference"].ne(0), "status"] = "mismatch"
print(checks["status"].value_counts())
checks.loc[checks["status"] != "match"].head(10)


status
match    380
Name: count, dtype: int64


,source_sheet,source_row,model_group,measure,monthly_sum,months_present,source_total,difference,status


### Save data and the audit together
Do not overwrite the source workbook. Review mismatches and conversion issues before using the table for analysis. Preserve blank measurement rows for audit; Lec 1's original script excluded rows with no numeric months, so its row count may differ.

The workbook labels include FY 2025–26. A worksheet name and twelve cells do not establish reporting completeness. Consult source notes before interpreting zeros or annual comparisons. Manufacturer mapping is deferred: combined model groups remain combined.


In [46]:
output = Path("data")
output.mkdir(parents=True, exist_ok=True)
all_data.to_csv(output / "passenger_cars_long.csv", index=False, date_format="%Y-%m-%d")
checks.to_csv(output / "validation.csv", index=False)
conversion_issues.to_csv(output / "conversion_issues.csv", index=False)
workbook.close()
print(output)


data
